# Bronze - State of Data Brasil (Tech Challenge)

Lê os CSVs da **Landing** (`landing/YYYY/`, edições 2023, 2024 e 2025), padroniza os nomes das colunas, trata os dados e grava na **Bronze** em Parquet particionado por `ano_pesquisa`:

| Tabela (pasta em `bronze/`) | Conteúdo |
|---|---|
| `respondentes` | 1 linha por respondente, campos de perfil/trabalho com nomes iguais em todos os anos |
| `respostas_multipla` | formato longo das perguntas de múltipla escolha: `id_respondente, pergunta, opcao, selecionado` |
| `dim_perguntas` | dicionário: coluna original -> nome na Bronze (linhagem/metadados) |

Reprocessar um ano só substitui a partição daquele ano (`partitionOverwriteMode=dynamic`).

In [ ]:
%idle_timeout 30
%worker_type G.1X
%number_of_workers 2

In [ ]:
import re, unicodedata
from pyspark.sql import functions as F

BUCKET = "tech-challenge-datahackers"
BASE   = f"s3://{BUCKET}"
ANOS   = [2023, 2024, 2025]      # anos a processar

spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

## 1. Funções de padronização

- Campos comuns viram `slug(rótulo)` (sem o código da pergunta), porque **o código muda entre os anos** (ex.: banco de dados é `4.g` em 2024 e `4.d` em 2025).
- Opções de múltipla escolha (colunas 0/1) viram linhas em `respostas_multipla`, identificadas por `pergunta` + `opcao`.
- `ALIAS_PERGUNTA` junta perguntas que mudaram de nome entre os anos.
- **2023** chega com outro cabeçalho: tuplas como `('P2_h ', 'Faixa salarial')` e rótulos por extenso. `normalizar_cabecalho` converte para o padrão `2.h_<rótulo>` e `ROTULOS_2023` troca o rótulo pelo de 2024 quando o código é a mesma pergunta/opção. Assim a Bronze tem os mesmos nomes de campo, pergunta e opção nos três anos.

In [ ]:
def slug(txt, maxlen=60):
    t = unicodedata.normalize("NFKD", txt).encode("ascii", "ignore").decode()
    t = re.sub(r"[^a-zA-Z0-9]+", "_", t).strip("_").lower()
    t = re.sub(r"_?em_20\d{2}$", "", t)          # remove o ano do nome (ex.: layoff_em_2025)
    return t[:maxlen].rstrip("_")

COD = re.compile(r"^(\d+)((?:\.[a-z0-9]+)*)[_ ](.*)$", re.S)

def parse_coluna(c):
    m = COD.match(c)
    if not m:
        return None, None, c
    return m.group(1) + m.group(2), m.group(1), m.group(3)    # codigo, secao, rotulo

# perguntas que mudaram de nome entre os anos (2025 -> nome canônico de 2024)
ALIAS_PERGUNTA = {
    "s4_linguagem_preferida": "s4_linguagem_de_programacao_dia_a_dia",
}

# campos que viram boolean / inteiro
BOOL_COLS = ["vive_no_brasil", "vive_no_estado_de_formacao", "atua_como_gestor",
             "satisfeito_atualmente", "possui_data_lake", "possui_data_warehouse"]
INT_COLS  = ["idade"]

# erros de digitação encontrados nas pesquisas de 2025 e 2023
CORRECOES = {
    "faixa_salarial": {"de R$ 25.001/mês a R$ 3000/mês": "de R$ 25.001/mês a R$ 30.000/mês",
                       "de R$ 101/mês a R$ 2.000/mês": "de R$ 1.001/mês a R$ 2.000/mês"},
    "numero_de_funcionarios": {"de 501 a 100": "de 501 a 1000"},
}

# --- 2023: cabeçalho no formato "('P2_h ', 'Faixa salarial')" ------------------------------------------
TUPLA_2023  = re.compile(r"^\('P(\d+)((?:_[a-z0-9]+)*)\s*',\s*'(.*?)'?\)$", re.S)
CODIGO_2023 = {"7.1": "7.a", "8.3": "8.c"}      # pergunta-mãe com código diferente do das suas opções

# 2023 -> rótulo de 2024 para o mesmo código de pergunta/opção (só onde o rótulo de 2023 é diferente).
# As opções 2.l (motivos de insatisfação) vêm em outra ordem e redação: foram alinhadas pelo significado.
ROTULOS_2023 = {
    '0': 'token',  # id da resposta
    '1.j': 'vive_no_estado_de_formacao',  # "Mudou de Estado?": mesma distribuição de vive_no_estado_de_formacao (1 = mora no estado em que se formou)
    '1.k': 'estado_de_origem',  # "Regiao de origem": na verdade traz o estado (ex.: "Acre (AC)")
    '2.a': 'situação_de_trabalho',
    '2.d': 'atua_como_gestor',
    '2.i': 'tempo_de_experiencia_em_dados',
    '2.j': 'tempo_de_experiencia_em_ti',
    '2.k': 'satisfeito_atualmente',
    '2.l': 'motivo_insatisfacao',
    '2.m': 'participou_de_entrevistas_ultimos_6m',
    '2.n': 'planos_de_mudar_de_emprego_6m',
    '2.o': 'criterios_para_escolha_de_emprego',
    '2.q': 'empresa_passou_por_layoff',
    '2.r': 'modelo_de_trabalho_atual',
    '2.s': 'modelo_de_trabalho_ideal',
    '2.t': 'atitude_em_caso_de_retorno_presencial',
    '3.a': 'numero_de_pessoas_em_dados',
    '3.b': 'cargos_no_time_de_dados_da_empresa',
    '3.c': 'responsabilidades_como_gestor',
    '3.d': 'desafios_como_gestor',
    '3.e': 'ai_generativa_e_llm_é_uma_prioridade?',
    '3.f': 'tipo_de_uso_de_ai_generativa_e_llm_na_empresa',
    '3.g': 'motivos_para_não_usar_ai_generativa_e_llm',
    '4.a': 'funcao_de_atuacao',
    '4.a.1': 'atuacao_em_dados',
    '4.b': 'fontes_de_dados_(dia_a_dia)',
    '4.c': 'fonte_de_dado_mais_usada',
    '4.d': 'linguagem_de_programacao_(dia_a_dia)',
    '4.e': 'linguagem_mais_usada',
    '4.f': 'linguagem_preferida',
    '4.g': 'banco_de_dados_(dia_a_dia)',
    '4.h': 'cloud_(dia_a_dia)',  # "Qual sua Cloud preferida?" (múltipla escolha) é a cloud do dia a dia; a preferida é a 4.i
    '4.j': 'ferramenta_de_bi_(dia_a_dia)',
    '4.k': 'ferramenta_de_bi_preferida',
    '4.l': 'tipo_de_uso_de_ai_generativa_e_llm_na_empresa',
    '4.m': 'usa_chatgpt_ou_copilot_no_trabalho?',
    '5.a': 'objetivo_na_area_de_dados',
    '5.b': 'oportunidade_buscada',
    '5.c': 'tempo_em_busca_de_oportunidade',
    '5.d': 'experiencia_em_processos_seletivos',
    '6.a': 'rotina_como_de',
    '6.b': 'ferramentas_etl_de',
    '6.c': 'possui_data_lake',
    '6.d': 'tecnologia_data_lake',
    '6.e': 'possui_data_warehouse',
    '6.f': 'tecnologia_data_warehouse',
    '6.g': 'ferramentas_de_qualidade_de_dados_(dia_a_dia)',
    '6.h': 'maior_tempo_gasto_como_de',
    '7.a': 'rotina_como_da',
    '7.b': 'ferramentas_etl_da',
    '7.c': 'ferramentas_autonomia_area_de_negocios',
    '7.d': 'maior_tempo_gasto_como_da',
    '8.a': 'rotina_como_ds',
    '8.b': 'tecnicas_e_metodos_ds',
    '8.c': 'tecnologias_ds',
    '8.d': 'maior_tempo_gasto_como_ds',
    '1.e.2': 'Sim, devido a minha Cor/Raça/Etnia',
    '1.e.3': 'Sim, devido a minha identidade de gênero',
    '1.e.4': 'Sim, devido ao fato de ser PCD',
    '1.f.7': 'Atenção dada pelas pessoas diante das minhas opiniões e ideias',
    '1.f.8': 'Relação com outras pessoas da empresa, em momentos de trabalho',
    '1.f.9': 'Relação com outras pessoas da empresa, em momentos de integração e outros momentos fora do trabalho',
    '2.o.6': 'Oportunidade de aprendizado e trabalhar com referências',
    '2.o.7': 'Plano de carreira e oportunidades de crescimento',
    '3.c.1': 'Pensar na visão de longo prazo de dados',
    '3.c.2': 'Organização de treinamentos e iniciativas',
    '3.c.3': 'Atração, seleção e contratação',
    '3.c.4': 'Decisão sobre contratação de ferramentas',
    '3.c.5': 'gestor da equipe de engenharia de dados',
    '3.c.6': 'gestor da equipe de estudos, relatórios',
    '3.c.7': 'gestor da equipe de Inteligência Artificial e Machine Learning',
    '3.c.8': 'Apesar de ser gestor ainda atuo na parte técnica',
    '3.c.9': 'Gestão de projetos de dados',
    '3.c.10': 'Gestão de produtos de dados',
    '3.c.11': 'Gestão de pessoas',
    '3.d.1': 'Contratar talentos',
    '3.d.2': 'Reter talentos',
    '3.d.3': 'Convencer a empresa a aumentar investimentos',
    '3.d.4': 'Gestão de equipes no ambiente remoto',
    '3.d.5': 'Gestão de projetos envolvendo áreas multidisciplinares',
    '3.d.6': 'Organizar as informações com qualidade e confiabilidade',
    '3.d.7': 'Processar e armazenar um alto volume de dados',
    '3.d.8': 'Gerar valor para as áreas de negócios',
    '3.d.9': 'Desenvolver e manter modelos Machine Learning em produção',
    '3.d.10': 'Gerenciar a expectativa das áreas',
    '3.d.11': 'Garantir a manutenção dos projetos e modelos em produção',
    '3.d.12': 'Conseguir levar inovação para a empresa',
    '3.d.13': 'Garantir (ROI) em projetos de dados',
    '3.f.4': 'AI Generativa e LLMs para melhorar produtos externos para os clientes finais',
    '4.d.15': 'Não utilizo nenhuma das linguagens listadas',
    '7.a.9': 'Utilizo ferramentas avançadas de estatística como SAS, SPSS, Stata etc, para realizar análises de dados.',
    '7.d.8': 'Desenvolvendo/cuidando da manutenção de planilhas para atender as áreas de negócio.',
    '8.d.2': 'Coletando e limpando dos dados que uso para análise e modelagem.',
    '2.l.1': 'Oportunidades de crescimento',  # 2023: Falta de oportunidade de crescimento no emprego atual
    '2.l.2': 'Remuneração/Salário',  # 2023: Salário atual não corresponde ao mercado
    '2.l.3': 'Relação com os gestores e líderes',  # 2023: Não tenho uma boa relação com meu líder/gestor
    '2.l.4': 'Gostaria de trabalhar em outra área',  # 2023: Gostaria de trabalhar em em outra área de atuação
    '2.l.5': 'Benefícios',  # 2023: Gostaria de receber mais benefícios
    '2.l.6': 'Ambiente e clima de trabalho',  # 2023: O clima de trabalho/ambiente não é bom
    '2.l.7': 'Maturidade da empresa em termos de tecnologia e dados',  # 2023: Falta de maturidade analítica na empresa
}

def normalizar_cabecalho(c):
    """('P2_h ', 'Faixa salarial') -> '2.h_faixa_salarial'. Colunas de 2024/2025 passam sem mudança."""
    m = TUPLA_2023.match(c.strip())
    if not m:
        return c
    cod = m.group(1) + m.group(2).replace("_", ".")
    cod = CODIGO_2023.get(cod, cod)
    return f"{cod}_{ROTULOS_2023.get(cod, m.group(3).strip())}"

In [ ]:
def montar_esquema(landing):
    """Classifica cada coluna do CSV e define o nome final na Bronze."""
    tipos = dict(landing.dtypes)
    info = {}
    for c in landing.columns:
        codigo, secao, rotulo = parse_coluna(c)
        info[c] = {"codigo": codigo, "secao": secao, "rotulo": rotulo}

    por_codigo = {}
    for c, v in info.items():
        if v["codigo"]:
            assert v["codigo"] not in por_codigo, f"codigo repetido: {v['codigo']}"
            por_codigo[v["codigo"]] = c

    numericos = ("tinyint", "smallint", "int", "bigint", "float", "double")
    flags = [c for c, v in info.items()
             if v["codigo"] and re.match(r"^\d+\.[a-z]\.\d+$", v["codigo"]) and tipos[c] in numericos]

    pais_de_flags = set()
    for c in flags:
        p = por_codigo.get(info[c]["codigo"].rsplit(".", 1)[0])
        if p:
            pais_de_flags.add(p)

    esquema = []     # (coluna_original, novo_nome, tipo, pergunta, opcao, rotulo)
    usados = set()
    for i, c in enumerate(landing.columns):
        v = info[c]
        if c in flags:
            pcod = v["codigo"].rsplit(".", 1)[0]
            pai = por_codigo.get(pcod)
            rot_pai = info[pai]["rotulo"] if pai else pcod
            pergunta = f"s{v['secao']}_{slug(rot_pai, 50)}"
            pergunta = ALIAS_PERGUNTA.get(pergunta, pergunta)
            esquema.append((c, f"flag_{i}", "flag", pergunta, slug(v["rotulo"]), v["rotulo"]))
        elif c in pais_de_flags:
            continue                                  # lista em texto: redundante com os flags
        else:
            nome = "id_respondente" if slug(v["rotulo"]) == "token" else slug(v["rotulo"])
            if nome in usados:
                nome = f"{nome}_{v['codigo'].replace('.', '_')}"
            usados.add(nome)
            esquema.append((c, nome, "campo", None, None, v["rotulo"]))
    return esquema

In [ ]:
def processar_ano(ano):
    landing = (spark.read.option("header", True).option("sep", ",").option("quote", '"')
           .option("escape", '"').option("multiLine", True).option("encoding", "UTF-8")
           .option("inferSchema", True).csv(f"{BASE}/landing/{ano}/"))
    originais = landing.columns
    landing = landing.toDF(*[normalizar_cabecalho(c) for c in originais])
    origem = dict(zip(landing.columns, originais))       # nome normalizado -> cabeçalho do CSV (linhagem)
    print(f"[{ano}] bronze: {landing.count()} linhas x {len(landing.columns)} colunas")

    esquema = montar_esquema(landing)
    sel = [F.col(f"`{c}`").alias(novo) for (c, novo, *_ ) in esquema]
    df = landing.select(*sel)

    flags  = [(novo, perg, op) for (_, novo, tipo, perg, op, _) in esquema if tipo == "flag"]
    campos = [novo for (_, novo, tipo, *_ ) in esquema if tipo == "campo"]

    # --- limpeza ---
    df = df.dropDuplicates(["id_respondente"]).withColumn("ano_pesquisa", F.lit(ano))
    if "data_hora_envio" in df.columns:                  # 2023 não tem data/hora de envio
        df = df.withColumn("data_hora_envio", F.to_timestamp("data_hora_envio", "dd/MM/yyyy HH:mm:ss"))

    for col, mapa in CORRECOES.items():
        if col in df.columns:
            df = df.replace(mapa, subset=[col])

    for c in BOOL_COLS:
        if c in df.columns:
            s = F.col(c).cast("string")
            df = df.withColumn(c, F.when(s.isin("True", "true", "1", "1.0"), F.lit(True))
                                   .when(s.isin("False", "false", "0", "0.0"), F.lit(False)))
    for c in INT_COLS:
        if c in df.columns:
            df = df.withColumn(c, F.col(c).cast("int"))

    # faixa salarial -> minimo, maximo, ponto medio
    if "faixa_salarial" in df.columns:
        limpo = F.regexp_replace("faixa_salarial", r"\.", "")
        primeiro = F.regexp_extract(limpo, r"(\d+)", 1)
        segundo  = F.regexp_extract(limpo, r"a R\$ (\d+)", 1)
        menos    = F.col("faixa_salarial").startswith("Menos")
        df = (df.withColumn("salario_min", F.when(menos, F.lit(0)).otherwise(primeiro.cast("int")))
                .withColumn("salario_max", F.when(menos, primeiro.cast("int"))
                                            .otherwise(F.when(segundo != "", segundo.cast("int"))))
                .withColumn("salario_medio", F.coalesce((F.col("salario_min") + F.col("salario_max")) / 2,
                                                        F.col("salario_min"))))

    # demais campos de texto: tipo string (evita conflito de tipos entre anos)
    tratados = set(BOOL_COLS + INT_COLS + ["data_hora_envio", "ano_pesquisa", "salario_min",
                                           "salario_max", "salario_medio"])
    for c in campos:
        if c not in tratados:
            df = df.withColumn(c, F.col(c).cast("string"))

    # --- tabela de respondentes e tabela longa ---
    resp = df.select("id_respondente", "ano_pesquisa",
                     *[c for c in df.columns if c in campos and c != "id_respondente"],
                     *[c for c in ["salario_min", "salario_max", "salario_medio"] if c in df.columns])

    estruturas = [F.struct(F.lit(p).alias("pergunta"), F.lit(o).alias("opcao"),
                           F.col(n).cast("int").alias("selecionado")) for (n, p, o) in flags]
    longo = (df.select("id_respondente", "ano_pesquisa", F.explode(F.array(*estruturas)).alias("t"))
               .select("id_respondente", "ano_pesquisa", "t.pergunta", "t.opcao", "t.selecionado")
               .where(F.col("selecionado").isNotNull()))

    # --- dicionario (linhagem) ---
    linhas = [(ano, origem.get(c, c), (novo if tipo == "campo" else f"{perg}/{op}"), tipo, perg, op, rot)
              for (c, novo, tipo, perg, op, rot) in esquema]
    dim = spark.createDataFrame(
        linhas, "ano_pesquisa int, coluna_original string, coluna_bronze string, tipo string, "
                "pergunta string, opcao string, rotulo_original string")
    return resp, longo, dim

## 2. Executar e gravar na Bronze

In [ ]:
def gravar(df, nome):
    df.write.mode("overwrite").partitionBy("ano_pesquisa").parquet(f"{BASE}/bronze/{nome}/")

for ano in ANOS:
    resp, longo, dim = processar_ano(ano)
    gravar(resp,  "respondentes")
    gravar(longo, "respostas_multipla")
    gravar(dim,   "dim_perguntas")
    print(f"[{ano}] bronze gravada")

## 3. Validação

Confira: nº de respondentes por ano (deve ser menor ou igual ao da Bronze, por causa das duplicatas) e a % de uso de cada linguagem,
calculada **só entre quem viu a pergunta** (as colunas 0/1 vazias significam "pergunta não exibida").

In [ ]:
resp  = spark.read.parquet(f"{BASE}/bronze/respondentes/")
longo = spark.read.parquet(f"{BASE}/bronze/respostas_multipla/")

resp.groupBy("ano_pesquisa").count().orderBy("ano_pesquisa").show()

(longo.where(F.col("pergunta") == "s4_linguagem_de_programacao_dia_a_dia")
      .groupBy("ano_pesquisa", "opcao")
      .agg(F.countDistinct("id_respondente").alias("nu_resp"),
           F.round(F.avg("selecionado") * 100, 1).alias("%_usa"))
      .where(F.col("opcao").isin("sql", "python", "r"))
      .orderBy("opcao", "ano_pesquisa").show())